# Shared EDA — Music Streaming Q2

Exploratory overview of the **shared foundation** (Section 5 of the blueprint).
It reads the artifacts produced by the data pipeline and reproduces the key
figures from `docs/GROUNDWORK_FINDINGS.md` as inline charts.

**Prerequisites** — run the pipeline first so the parquet files exist:
```bash
bash src/data/run_user_window_agg.sh "../Dataset/Raw_Data.zip"
python -m src.data.build_labels
```
and install plotting deps: `pip3 install matplotlib jupyter`.


## Setup


In [ ]:
import sys
from pathlib import Path
# make `import src` work when running from the notebooks/ folder
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from src import config

DER = config.DERIVED_DIR
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True, "grid.alpha": 0.3})
BLUE, RED = "#2E5C8A", "#C0392B"

agg = pl.read_parquet(DER / "user_window_agg.parquet")        # all users, both windows
mt  = pl.read_parquet(DER / "user_modeling_table.parquet")    # study population + label + split
print(f"all users:   {agg.height:,}")
print(f"study users: {mt.height:,}")
print(f"windows: early={config.EARLY_WINDOW_DAYS}  label={config.LABEL_WINDOW_DAYS}")


## 1. Window overlap (survivorship)

Who appears in the early window (features), the label window (outcome), or both.
Only users in **both** enter the study population.


In [ ]:
both  = int(((agg["e_impr"] > 0) & (agg["l_impr"] > 0)).sum())
early = int(((agg["e_impr"] > 0) & (agg["l_impr"] == 0)).sum())
label = int(((agg["e_impr"] == 0) & (agg["l_impr"] > 0)).sum())

fig, ax = plt.subplots()
bars = ax.bar(["both\n(study pop.)", "early only", "label only"],
              [both, early, label], color=[BLUE, "#7BA7CC", RED])
ax.bar_label(bars, fmt="{:,.0f}")
ax.set_ylabel("users"); ax.set_title("Window overlap")
plt.show()
print(f"study population = {both:,} ({both/agg.height:.0%} of all users)")


## 2. Activeness threshold sensitivity

`inactive` = label-window click rate ≤ threshold. The inactive share is very
sensitive to the cutoff — a definition the team must choose deliberately.


In [ ]:
ths = [0.0, 0.01, 0.05, 0.10]
rates = [float((mt["l_click_rate"] <= t).mean()) for t in ths]

fig, ax = plt.subplots()
ax.plot([f"<= {t}" for t in ths], rates, marker="o", color=BLUE)
for x, r in zip(range(len(ths)), rates):
    ax.annotate(f"{r:.1%}", (x, r), textcoords="offset points", xytext=(0, 8))
ax.set_ylim(0, 1); ax.set_ylabel("inactive share")
ax.set_xlabel("click-rate threshold"); ax.set_title("Inactive share vs. threshold")
plt.show()
print("primary threshold in config:", config.ACTIVE_CLICK_THRESHOLD)


## 3. Class balance


In [ ]:
inactive_rate = float(mt["is_inactive"].mean())
fig, ax = plt.subplots(figsize=(4, 4))
ax.pie([inactive_rate, 1 - inactive_rate], labels=["inactive", "active"],
       autopct="%1.1f%%", colors=[RED, BLUE], startangle=90)
ax.set_title(f"Class balance (threshold ≤ {config.ACTIVE_CLICK_THRESHOLD})")
plt.show()


## 4. Active vs. inactive — early behaviour

Mean early-window features by class. Volume and breadth of early exposure
separate the classes strongly — a good sign Part (b) is learnable.


In [ ]:
feat_means = ["e_impr", "e_clicks", "e_active_days", "e_avg_view_time"]
g = (mt.group_by("is_inactive")
       .agg([pl.col(f).mean().alias(f) for f in feat_means])
       .sort("is_inactive"))
active = g.filter(~pl.col("is_inactive")).select(feat_means).row(0)
inact  = g.filter(pl.col("is_inactive")).select(feat_means).row(0)

x = np.arange(len(feat_means)); w = 0.38
fig, ax = plt.subplots()
ax.bar(x - w/2, active, w, label="active", color=BLUE)
ax.bar(x + w/2, inact,  w, label="inactive", color=RED)
ax.set_xticks(x); ax.set_xticklabels(feat_means, rotation=20)
ax.set_ylabel("mean"); ax.set_title("Early-window means by class"); ax.legend()
plt.show()


## 5. Distribution of early engagement

`e_impr` is heavy-tailed, so we plot it on a log axis. Active users sit clearly
to the right (more early impressions).


In [ ]:
fig, ax = plt.subplots()
for v, lab, col in [(False, "active", BLUE), (True, "inactive", RED)]:
    x = mt.filter(pl.col("is_inactive") == v)["e_impr"].to_numpy()
    ax.hist(np.log1p(x), bins=40, alpha=0.5, label=lab, color=col, density=True)
ax.set_xlabel("log(1 + early impressions)"); ax.set_ylabel("density")
ax.set_title("Early-impression distribution by class"); ax.legend()
plt.show()


## 6. Baseline model (the number to beat)

Gradient boosting on early-window features only (leakage-safe). Members B and C
should report against this on the same split.


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score

FEATS = ["e_impr","e_clicks","e_likes","e_shares","e_comments","e_viewcomment",
         "e_homepage","e_active_days","e_avg_pos","e_click_rate","e_like_rate","e_avg_view_time"]
assert not (set(FEATS) & config.FORBIDDEN_FEATURES), "leakage guard"
d = mt.with_columns([pl.col(c).fill_null(0.0) for c in FEATS])

def XY(s):
    dd = d.filter(pl.col("split") == s)
    return dd.select(FEATS).to_numpy(), dd["is_inactive"].to_numpy().astype(int)

Xtr, ytr = XY("train"); Xte, yte = XY("test")
clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.07, max_depth=6,
                                     l2_regularization=1.0, random_state=42).fit(Xtr, ytr)
p = clf.predict_proba(Xte)[:, 1]
auc, ap = roc_auc_score(yte, p), average_precision_score(yte, p)
print(f"ROC-AUC = {auc:.4f}   PR-AUC = {ap:.4f}   base rate = {yte.mean():.3f}")


In [ ]:
fpr, tpr, _ = roc_curve(yte, p)
prec, rec, _ = precision_recall_curve(yte, p)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(fpr, tpr, color=BLUE, label=f"AUC={auc:.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="grey")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].set_title("ROC"); axes[0].legend()
axes[1].plot(rec, prec, color=RED, label=f"AP={ap:.3f}")
axes[1].axhline(yte.mean(), ls="--", color="grey", label="base rate")
axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision"); axes[1].set_title("Precision-Recall"); axes[1].legend()
plt.tight_layout(); plt.show()


In [ ]:
# Permutation importance on a test subsample (fast)
from sklearn.inspection import permutation_importance
rng = np.random.default_rng(0)
idx = rng.choice(len(yte), size=min(20000, len(yte)), replace=False)
pi = permutation_importance(clf, Xte[idx], yte[idx], n_repeats=3,
                            scoring="roc_auc", random_state=0, n_jobs=4)
order = np.argsort(pi.importances_mean)
fig, ax = plt.subplots()
ax.barh([FEATS[i] for i in order], pi.importances_mean[order], color=BLUE)
ax.set_xlabel("ROC-AUC drop when shuffled"); ax.set_title("Feature importance")
plt.tight_layout(); plt.show()


## Takeaways

- **~72%** of study users are inactive (zero clicks) — heavy imbalance; track PR-AUC.
- Only **~26%** of users span both windows; the rest are excluded (survivorship).
- Active users have far more early impressions/clicks — early signal is real.
- Baseline **ROC-AUC ≈ 0.71** is the bar for Members B and C.

See `docs/GROUNDWORK_FINDINGS.md` for the written version and `docs/Thesis_Blueprint_Q2.docx` for the full plan.
